# Quantum Chemistry with ORCA
## Introduction

In this notebook, we will carry out quantum chemical calculations on organic and inorganic molecules. 
- We will use the ORCA program package developed at the Max-Planck-Institut für Kohlenforschung.
- ORCA is a general-purpose quantum chemistry package initiated by Frank Neese in 1999.
- ORCA is free for academic use and features a variety of quantum chemical methods.
- We will be using Density Functional Theory (DFT) and Gaussian-Type basis sets.

## Learning outcomes

After completing this notebook, you will be able to:
- Understand the key parts of ORCA input files for molecular systems.
- Optimize molecular structures with ORCA.
- Compare total energies of molecular isomers to find out the lowest-energy isomer.
- Calculate and interpret infrared vibrational spectra for molecules.

## Instructions for using the notebook
- The notebook is composed of cells and the blue highlight in the left side of the notebook shows which cell is currently active. 
- You can run the active cell by clicking the "play" button from the toolbar above ("Run this cell and advance", keyboard shortcut is Shift + Enter).
- The notebook includes instruction cells like this cell and gray cells that contain Python code.
- When you run a cell and see the symbol `In [*]` in the top left corner of the cell, the notebook is executing code. Please wait until the symbol changes to a number like `[1]`.
- Run the cells from the top to the bottom.
- The files generated during the exercise are visible in the file browser on the left side of the notebook. Text files (.inp, .xyz, and .out extension) can be opened for viewing by double-clicking them.
- If something goes totally wrong, you can open the Kernel menu and choose "Restart Kernel and Clear Outputs of All Cells". This will reset the state of the notebook and you can start from the beginning.

## Setting up the calculation environment for ORCA

Run the next code block to setup the calculation environment for ORCA.

In [ ]:
import sys
sys.path.append('../tools')
from qctools import load_xyz, load_xyz_as_traj, show_molecule

## Writing files and visualizing molecular structures

Let's start by visualizing a molecular structure using NGLview tool. We first need to create a molecular structure file. 
- We can write files in notebooks with the `%%writefile` command (a so-called IPython magic command available in notebooks).
- The file format is so-called XYZ file. This is a standard human-readable file format in quantum chemistry. It contains the cartesian coordinates of each atom in format `symbol X Y Z`).

Run the following code block to create an XYZ file with a distorted water molecular (H-O-H angle of 60 degrees, O-H distances of 1.2 Å).

In [ ]:
%%writefile h2o_dist.xyz
3
Distorted water molecule
 O                  0.00000000    0.00000000   -0.16627688
 H                  0.00000000   -0.60000000    0.87295361
 H                 -0.00000000    0.60000000    0.87295361

Next, run the following code block to load the linear water molecule in an NGLview window.

In [ ]:
mol = load_xyz('h2o_dist.xyz')
show_molecule(mol)

## Interacting with the molecular structure
You can interact with the molecular structure in the above view as follows:
- Rotate: hold the left mouse button and move the mouse.
- Zoom: mouse scrollwheel.
- Translate: hold the right mouse button and move the mouse.
  
Next, we will optimize the distorted structure to a reasonable minimum. 

## Structure optimization

In quantum chemical molecular modelling, the first task is usually the structural optimization of the studied system. This means that we will find the lowest-energy three-dimensional structure for the molecule.

Standard structure optimization methods can find local minima on the potential energy surface. This means that the optimization algorithm does not search all possible conformations of the molecule. We will have a look into conformation analysis later.

Let's optimize the structure of water molecule with Density Functional Theory (DFT). We will use the DFT-PBE functional, which is the most commonly used GGA-level DFT functional. We will look into more accurate DFT methods later, but for typical main group compounds DFT-PBE performs 

Let's start by writing an ORCA input file. The input file shown in the code block below consists of following items:
- The first line in the code block is the `%%writefile` command that writes the following lines in file `h2o_opt.inp`. The convention is that ORCA input files have file extension `.inp`.
- The line starting with `!` is so-called ORCA simple input line. Here we define the DFT-PBE method (`PBE`) and the one-electron basis set (`DEF2-TZVP`). The `Opt` keyword means that we want to optimize the molecular structure.
- The simple input line is followed by the XYZ coordinates of the water molecule. The line `xyz 0 1` tells ORCA that we will give the molecular structure in XYZ format, the charge of the molecule is 0 and the spin multiplicity is 1 (singlet, no unpaired electrons).
- The calculation will use only one CPU core because the system is so small. Later, we will add more CPUs (computing power).

Run the next code block to create the input file.

In [ ]:
%%writefile h2o_opt.inp
!PBE DEF2-TZVP Opt
* xyz 0 1
 O                  0.00000000    0.00000000   -0.16627688
 H                  0.00000000   -0.60000000    0.87295361
 H                 -0.00000000    0.60000000    0.87295361
*

On a general level, the structural optimization will proceed as follows:

- ORCA calculates the electronic structure (molecular orbitals) of the initial structure. This also yields the electronic total energy of the system.
- ORCA calculates the forces affecting the atoms (first derivatives of the total energy with respect to atomic coordinates).
- ORCA displaces the atoms in such way that the forces affecting the atoms are reduced. The goal is to reach a local minimum structure where all forces are zero (in practice, close to zero).
- The above three steps are repeated until the forces affecting the atoms are below the default convergence criteria in ORCA. These are relatively strict, and can be changed by the user.
- After the convergence criteria have been reached, the structure optimization is complete.

Run the next code block to run the structure optimization. The job should take less than 30 seconds to finish.

In [ ]:
!$ORCA_HOME/orca h2o_opt.inp > h2o_opt.out
!grep "TERMINATED" h2o_opt.out
!grep "TOTAL RUN TIME" h2o_opt.out

Let's load all intermediate structures of the optimization to see how it proceeded (this is often called the "trajectory"). 

Run the following code block, which loads the optimization trajectory. After loading, the view shows the initial structure. Move your mouse  over the view and press play to see how the optimization changed the molecular structure.

In [ ]:
mol2 = load_xyz_as_traj('h2o_opt_trj.xyz')
show_molecule(mol2)

Let's see how the total energy of the system changed during the optimization. The next code block will read the total electronic energies and plot them with Matplotlib. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

energies_raw = !grep "FINAL SINGLE POINT ENERGY" h2o_opt.out | awk '{print $NF}'
energies = np.asarray(energies_raw, dtype=float)
steps = np.arange(1, len(energies_raw) + 1)

plt.plot(steps, energies, marker = 'o', color = 'black', label = 'Total energy')
plt.xlabel('Optimization step')
plt.ylabel('Total energy (Hartree)')
plt.xlim(1, len(steps) + 1)
plt.ylim(np.min(energies)*1.0001, np.max(energies))
plt.xticks(np.arange(1, len(steps) + 1, 1))
plt.yticks(np.linspace(np.min(energies), max(energies), 6))
plt.legend(loc = 'upper right')
plt.title('Optimization of H$_2$O')
plt.show()

The plot above shows how the total energy decreases and then no longer changes. In chemistry, we are usually more interested in relative energies. Let's see how the same plot looks when we set the energy of the optimized structure as zero and calculate the relative energies of the intermediate structures.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

HARTREE_TO_KJMOL = 2625.5
energies_raw = !grep "FINAL SINGLE POINT ENERGY" h2o_opt.out | awk '{print $NF}'
energies = np.asarray(energies_raw, dtype=float)
rel_energies = (energies - np.min(energies)) * HARTREE_TO_KJMOL
steps = np.arange(1, len(energies_raw) + 1)

plt.plot(steps, rel_energies, marker = 'o', color = 'red', label = 'Relative energy')
plt.xlabel('Optimization step')
plt.ylabel('Relative energy (kJ/mmol)')
plt.xlim(1, len(steps) + 1)
plt.ylim(0, np.max(rel_energies))
plt.xticks(np.arange(1, len(steps) + 1, 1))
plt.yticks(np.linspace(np.min(rel_energies), max(rel_energies), 6))
plt.legend(loc = 'upper right')
plt.title('Optimization of H$_2$O')
plt.show()

The plot above shows that the starting structure was indeed a high-energy structure (224 kJ/mol higher in energy compared to the final minimum).

## Harmonic frequency calculation

*Explanation TODO*

In [ ]:
%%writefile h2o_fq.inp
!PBE DEF2-TZVP Freq
* xyzfile 0 1 h2o_opt.xyz

In [ ]:
!$ORCA_HOME/orca h2o_fq.inp > h2o_fq.out
!grep "TERMINATED" h2o_fq.out
!grep "TOTAL RUN TIME" h2o_fq.out

In [ ]:
!sed -n -e '/VIBRATIONAL FREQUENCIES/,/NORMAL MODES/p' h2o_fq.out | head -n -3

In [ ]:
!orca_pltvib h2o_fq.hess 6 7 8

In [ ]:
freq = load_xyz_as_traj('h2o_fq.hess.v006.xyz')
show_molecule(freq)